# TS Forecasting V8: Rule-Safe LGBM Hybrid

This notebook combines the two strongest public approaches:
- `hedge-fund-feature-analysis-lightgbm.ipynb`
- `rule-safe-freq-encoding-lgbm-pipeline-cv-0-25.ipynb`

Design goals:
- strict time-safe pipeline (`ts_index` split)
- per-horizon specialists (`1, 3, 10, 25`)
- frequency and target encodings built only from training portion
- cross-sectional and temporal features
- train+test continuity for lag/rolling/EWM features
- weighted post-calibration on local holdout


In [1]:
import gc
import os
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = 'train.parquet'
    TEST_PATH = 'test.parquet'

VAL_THRESHOLD = 3500
HORIZONS = [1, 3, 10, 25]
SEEDS = [42, 2024, 777, 1337, 9999]
APPLY_CALIBRATION = True

# Set to an integer for quick smoke runs locally; keep None for full training.
MAX_ROWS_DEBUG = None

print('Train path:', TRAIN_PATH)
print('Test path :', TEST_PATH)
print('VAL_THRESHOLD =', VAL_THRESHOLD)
print('HORIZONS =', HORIZONS)
print('SEEDS =', SEEDS)


Train path: /kaggle/input/competitions/ts-forecasting/train.parquet
Test path : /kaggle/input/competitions/ts-forecasting/test.parquet
VAL_THRESHOLD = 3500
HORIZONS = [1, 3, 10, 25]
SEEDS = [42, 2024, 777, 1337, 9999]


## Metric and Calibration

Competition metric:
`score = sqrt(1 - clip01(sum(w*(y-p)^2) / sum(w*y^2)))`


In [2]:
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))


def weighted_rmse_score(y_true, y_pred, w) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)

    denom = np.sum(w * (y_true ** 2))
    if denom <= 0:
        return 0.0
    ratio = np.sum(w * ((y_true - y_pred) ** 2)) / denom
    return float(np.sqrt(1.0 - _clip01(ratio)))


def weighted_linear_calibration(y_true, y_pred, w):
    """
    Fit y ~= a * pred + b with weighted least squares.
    """
    y = np.asarray(y_true, dtype=np.float64)
    p = np.asarray(y_pred, dtype=np.float64)
    ww = np.asarray(w, dtype=np.float64)

    wsum = ww.sum()
    if wsum <= 0:
        return 1.0, 0.0

    p_mean = np.sum(ww * p) / wsum
    y_mean = np.sum(ww * y) / wsum

    cov = np.sum(ww * (p - p_mean) * (y - y_mean))
    var = np.sum(ww * (p - p_mean) ** 2)

    if var <= 1e-12:
        return 1.0, 0.0

    a = cov / var
    b = y_mean - a * p_mean

    if not np.isfinite(a):
        a = 1.0
    if not np.isfinite(b):
        b = 0.0

    return float(a), float(b)


## Rule-Safe Encodings and Features

Important constraints:
- Encoding stats are computed only from `train[ts_index <= VAL_THRESHOLD]`.
- Temporal features are computed on `concat(train_h, test_h)` to preserve continuity at the train-test boundary.


In [3]:
def compute_train_stats(train_fit: pd.DataFrame) -> dict:
    stats = {}

    sub_code_freq = train_fit['sub_code'].value_counts()
    stats['sub_code_freq'] = sub_code_freq.to_dict()
    stats['sub_code_freq_rank'] = sub_code_freq.rank(ascending=False, method='dense').to_dict()

    stats['sub_code_target_mean'] = train_fit.groupby('sub_code')['y_target'].mean().to_dict()
    stats['sub_category_target_mean'] = train_fit.groupby('sub_category')['y_target'].mean().to_dict()
    stats['code_target_mean'] = train_fit.groupby('code')['y_target'].mean().to_dict()
    stats['global_target_mean'] = float(train_fit['y_target'].mean())

    return stats


def create_features_from_combined(df: pd.DataFrame, stats: dict, horizon: int) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(['code', 'sub_code', 'ts_index'], kind='mergesort').reset_index(drop=True)

    # Low-cardinality category one-hot
    subcat_dummies = pd.get_dummies(df['sub_category'], prefix='subcat', dtype=np.int8)
    df = pd.concat([df, subcat_dummies], axis=1)

    # Frequency + target means from fit split only
    df['sub_code_freq'] = df['sub_code'].map(stats['sub_code_freq']).fillna(1.0)
    df['sub_code_log_freq'] = np.log1p(df['sub_code_freq'])
    df['sub_code_freq_rank'] = df['sub_code'].map(stats['sub_code_freq_rank']).fillna(df['sub_code'].nunique())

    df['sub_code_mean_y'] = df['sub_code'].map(stats['sub_code_target_mean']).fillna(stats['global_target_mean'])
    df['sub_category_mean_y'] = df['sub_category'].map(stats['sub_category_target_mean']).fillna(stats['global_target_mean'])
    df['code_mean_y'] = df['code'].map(stats['code_target_mean']).fillna(stats['global_target_mean'])

    df['sub_code_ts_freq'] = df.groupby(['ts_index', 'sub_code'])['sub_code'].transform('count')
    df['sub_code_rel_freq'] = df['sub_code_ts_freq'] / (df['sub_code_freq'] + 1.0)

    # Core interactions
    df['spread_al_am'] = df['feature_al'] - df['feature_am']
    df['ratio_al_am'] = df['feature_al'] / (np.abs(df['feature_am']) + 1e-7)
    df['spread_cg_by'] = df['feature_cg'] - df['feature_by']
    df['ratio_cg_by'] = df['feature_cg'] / (np.abs(df['feature_by']) + 1e-7)
    df['spread_s_t'] = df['feature_s'] - df['feature_t']
    df['ratio_s_t'] = df['feature_s'] / (np.abs(df['feature_t']) + 1e-7)

    grp = df.groupby(['code', 'sub_code'], sort=False)

    # Lags and momentum
    lag_feats = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'spread_al_am', 'spread_cg_by']
    for f in lag_feats:
        if f not in df.columns:
            continue
        for lag in (1, 2, 5):
            df[f'{f}_lag{lag}'] = grp[f].shift(lag)
        df[f'{f}_diff1'] = grp[f].diff(1)
        df[f'{f}_pct1'] = grp[f].pct_change().replace([np.inf, -np.inf], 0.0)

    # Causal rolling / ewm
    ewm_spans = [3, 5] if horizon <= 3 else [5, 7]
    roll_windows = [5, 9]
    roll_feats = ['feature_al', 'feature_am', 'spread_al_am']
    for f in roll_feats:
        if f not in df.columns:
            continue
        shifted = grp[f].shift(1)
        shifted_grp = shifted.groupby([df['code'], df['sub_code']], sort=False)
        for w in roll_windows:
            df[f'{f}_rmean{w}'] = shifted_grp.transform(lambda x: x.rolling(w, min_periods=max(2, w // 3)).mean())
            df[f'{f}_rstd{w}'] = shifted_grp.transform(lambda x: x.rolling(w, min_periods=max(2, w // 3)).std())
        for span in ewm_spans:
            df[f'{f}_ewm{span}'] = shifted_grp.transform(lambda x: x.ewm(span=span, adjust=False).mean())

    # Cross-sectional normalization and robust ranks
    cs_feats = [
        'feature_al', 'feature_am', 'feature_cg', 'feature_by', 'feature_s', 'feature_t',
        'spread_al_am', 'spread_cg_by', 'spread_s_t'
    ]
    for f in cs_feats:
        if f not in df.columns:
            continue
        g = df.groupby('ts_index')[f]
        df[f'{f}_ts_std'] = g.transform('std')
        df[f'{f}_z'] = (df[f] - g.transform('mean')) / (g.transform('std') + 1e-7)
        df[f'{f}_rank'] = g.rank(pct=True)

    # Hierarchical deviation features (from feature-analysis notebook)
    hier_feats = ['feature_al', 'feature_am', 'spread_al_am']
    for key in ['sub_category', 'code']:
        for f in hier_feats:
            if f not in df.columns:
                continue
            hmean = df.groupby([key, 'ts_index'])[f].transform('mean')
            df[f'{f}_vs_{key}'] = df[f] - hmean

    # Time encoding
    df['ts_sin'] = np.sin(2.0 * np.pi * df['ts_index'] / 100.0)
    df['ts_cos'] = np.cos(2.0 * np.pi * df['ts_index'] / 100.0)

    return df.fillna(0.0)


## Training Loop (Per Horizon, Multi-Seed)

Default mode uses one rule-safe holdout:
- fit: `ts_index <= VAL_THRESHOLD`
- validation: `ts_index > VAL_THRESHOLD`


In [4]:
HORIZON_PARAMS = {
    1: dict(learning_rate=0.015, n_estimators=4200, num_leaves=70, min_child_samples=250, lambda_l2=12.0, early_stop=220),
    3: dict(learning_rate=0.015, n_estimators=4200, num_leaves=75, min_child_samples=225, lambda_l2=11.0, early_stop=200),
    10: dict(learning_rate=0.015, n_estimators=4200, num_leaves=85, min_child_samples=180, lambda_l2=9.0, early_stop=180),
    25: dict(learning_rate=0.015, n_estimators=4200, num_leaves=92, min_child_samples=150, lambda_l2=8.0, early_stop=170),
}

BASE_PARAMS = {
    'objective': 'regression',
    'metric': 'rmse',
    'feature_fraction': 0.6,
    'bagging_fraction': 0.7,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'verbosity': -1,
}


def train_one_horizon(horizon: int):
    print('\\n' + '=' * 72)
    print(f'HORIZON {horizon}')
    print('=' * 72)

    train_raw = pd.read_parquet(TRAIN_PATH).query('horizon == @horizon').copy()
    test_raw = pd.read_parquet(TEST_PATH).query('horizon == @horizon').copy()

    if MAX_ROWS_DEBUG is not None:
        train_raw = train_raw.sample(min(MAX_ROWS_DEBUG, len(train_raw)), random_state=42)
        test_raw = test_raw.sample(min(MAX_ROWS_DEBUG, len(test_raw)), random_state=42)

    fit_mask = train_raw['ts_index'] <= VAL_THRESHOLD
    val_mask = ~fit_mask

    if val_mask.sum() == 0:
        raise ValueError(f'No validation rows for horizon={horizon}. Increase train/test cutoff logic.')

    stats = compute_train_stats(train_raw.loc[fit_mask, ['code', 'sub_code', 'sub_category', 'y_target']])

    train_raw['_is_test'] = 0
    test_raw['_is_test'] = 1
    test_raw['y_target'] = np.nan
    test_raw['weight'] = np.nan

    combined = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)
    combined = create_features_from_combined(combined, stats, horizon)

    train = combined[combined['_is_test'] == 0].copy()
    test = combined[combined['_is_test'] == 1].copy()

    drop_cols = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target', '_is_test'}
    feats = [c for c in train.columns if c not in drop_cols]

    tr_m = train['ts_index'] <= VAL_THRESHOLD
    va_m = train['ts_index'] > VAL_THRESHOLD

    X_tr = train.loc[tr_m, feats]
    y_tr = train.loc[tr_m, 'y_target'].astype(np.float32)
    w_tr = train.loc[tr_m, 'weight'].astype(np.float32)

    X_va = train.loc[va_m, feats]
    y_va = train.loc[va_m, 'y_target'].astype(np.float32)
    w_va = train.loc[va_m, 'weight'].astype(np.float32)

    params = BASE_PARAMS.copy()
    cfg = HORIZON_PARAMS[horizon].copy()
    early_stop = cfg.pop('early_stop')
    params.update(cfg)

    print(f'rows train={len(X_tr):,}, val={len(X_va):,}, test={len(test):,}, feats={len(feats)}')

    val_pred = np.zeros(len(X_va), dtype=np.float64)
    test_pred = np.zeros(len(test), dtype=np.float64)

    for i, seed in enumerate(SEEDS, 1):
        print(f'  seed {i}/{len(SEEDS)}: {seed}')
        model = lgb.LGBMRegressor(**params, random_state=seed)
        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            eval_sample_weight=[w_va],
            callbacks=[lgb.early_stopping(early_stop, verbose=False)],
        )
        val_pred += model.predict(X_va) / len(SEEDS)
        test_pred += model.predict(test[feats]) / len(SEEDS)

    raw_score = weighted_rmse_score(y_va.values, val_pred, w_va.values)
    print(f'  raw holdout score: {raw_score:.6f}')

    a, b = 1.0, 0.0
    final_val_pred = val_pred
    if APPLY_CALIBRATION:
        a, b = weighted_linear_calibration(y_va.values, val_pred, w_va.values)
        cal_val_pred = a * val_pred + b
        cal_score = weighted_rmse_score(y_va.values, cal_val_pred, w_va.values)
        print(f'  calibration: a={a:.6f}, b={b:.6f}, score={cal_score:.6f}')
        if cal_score >= raw_score:
            final_val_pred = cal_val_pred
            test_pred = a * test_pred + b
            final_score = cal_score
        else:
            final_score = raw_score
    else:
        final_score = raw_score

    out_test = pd.DataFrame({'id': test['id'].values, 'prediction': test_pred.astype(np.float64)})

    out_val = pd.DataFrame({
        'id': train.loc[va_m, 'id'].values,
        'y_true': y_va.values,
        'y_pred': final_val_pred.astype(np.float64),
        'w': w_va.values,
        'horizon': horizon,
    })

    # Cleanup
    del train_raw, test_raw, combined, train, test, X_tr, y_tr, w_tr, X_va, y_va, w_va
    gc.collect()

    return out_test, out_val, final_score


## Run Training and Build Submission

This will train 4 horizon-specific ensembles and create:
- `submission_v8_rule_safe_hybrid.csv`
- `oof_v8_rule_safe_hybrid.csv`


In [5]:
all_test = []
all_val = []
per_h_scores = {}

for h in HORIZONS:
    test_part, val_part, score_h = train_one_horizon(h)
    all_test.append(test_part)
    all_val.append(val_part)
    per_h_scores[h] = score_h

submission = pd.concat(all_test, ignore_index=True)
submission.to_csv('submission_v8_rule_safe_hybrid.csv', index=False)

oof = pd.concat(all_val, ignore_index=True)
oof.to_csv('oof_v8_rule_safe_hybrid.csv', index=False)

agg_score = weighted_rmse_score(oof['y_true'].values, oof['y_pred'].values, oof['w'].values)

print('\\n' + '=' * 72)
print('Per-horizon local scores:')
for h in HORIZONS:
    print(f'  h={h:2d}: {per_h_scores[h]:.6f}')
print('-' * 72)
print(f'Aggregate holdout score: {agg_score:.6f}')
print('Saved: submission_v8_rule_safe_hybrid.csv')
print('Saved: oof_v8_rule_safe_hybrid.csv')
print('=' * 72)

submission.head()


\n========================================================================
HORIZON 1
rows train=1,351,193, val=43,460, test=379,617, feats=188
  seed 1/5: 42
  seed 2/5: 2024
  seed 3/5: 777
  seed 4/5: 1337
  seed 5/5: 9999
  raw holdout score: 0.068612
  calibration: a=1.440556, b=0.000006, score=0.072044
\n========================================================================
HORIZON 3
rows train=1,342,793, val=43,023, test=376,558, feats=188
  seed 1/5: 42
  seed 2/5: 2024
  seed 3/5: 777
  seed 4/5: 1337
  seed 5/5: 9999
  raw holdout score: 0.126975
  calibration: a=1.513832, b=0.000018, score=0.134926
\n========================================================================
HORIZON 10
rows train=1,296,269, val=40,967, test=362,057, feats=188
  seed 1/5: 42
  seed 2/5: 2024
  seed 3/5: 777
  seed 4/5: 1337
  seed 5/5: 9999
  raw holdout score: 0.221834
  calibration: a=1.407398, b=0.000013, score=0.231632
\n======================================================================

,id,prediction
0,10BAVIDU__07YQ9WA4__PZ9S1Z4V__1__4175,-0.006101
1,10BAVIDU__07YQ9WA4__V8BKY1IV__1__4175,0.003322
2,10BAVIDU__07YQ9WA4__DPPUO5X2__1__4175,-0.096895
3,10BAVIDU__07YQ9WA4__NQ58FVQM__1__4175,-0.089019
4,10BAVIDU__07YQ9WA4__PHHHVYZI__1__4175,-0.089244


## Notes

- This notebook is designed to run directly on Kaggle with competition data attached.
- If local runtime is long, start with `MAX_ROWS_DEBUG = 300000` for a smoke test.
- For stricter robustness, next version should switch from one holdout to walk-forward CV and tune per-horizon calibration from OOF folds.
